<a href="https://colab.research.google.com/github/viditmahla/FUTSA_project/blob/main/ERW_CDR_PyCO2SYS_vectorised.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌊 ERW / CDR Maximum Alkalinity Model — PyCO2SYS Edition

**Python port of `cdr_via_erw.R`** — replaces `seacarb::carb()` with `PyCO2SYS` and `seacarb::Kspc()` with Mucci (1983), exactly as configured in the companion *PyCO2SYS Batch Excel* notebook.

### What this notebook does
1. **Loads raw discharge data** (pH, Temp, Ca²⁺, Total Alkalinity, Salinity) — same format as the PyCO2SYS Batch notebook.
2. **Runs an initial PyCO2SYS pass** on every row to establish the baseline carbonate system (DIC, CO₃²⁻, pCO₂, Ω_calcite …).
3. **Iterates rock addition** (coarse j-loop → fine k-loop, identical logic to the R model) until calcite supersaturation reaches `OMEGA_THRESH` or the maximum addition is exhausted.
4. **Replaces `seacarb`** at every loop step with `PyCO2SYS` using:
   - `opt_k_carbonic = 14` (Millero 2010) — same as the batch notebook
   - `opt_pH_scale = 1` (Total scale)
   - Mucci (1983) Ksp passed explicitly (`k_calcite`, `k_aragonite`)
   - Measured Ca²⁺ via `total_calcium`
5. **Exports** a multi-sheet Excel: baseline PyCO2SYS outputs + ERW loop outputs.

---
### seacarb → PyCO2SYS translation
| R (seacarb) | Python (PyCO2SYS) |
|---|---|
| `carb(15, ALK, DIC, …)` | `pyco2.sys(par1=ALK, par1_type=1, par2=DIC, par2_type=2, …)` |
| `carb(24, pCO2, ALK, …)` | `pyco2.sys(par1=ALK, par1_type=1, par2=pCO2, par2_type=4, …)` |
| `Kspc(S, T, P)` | `mucci1983_KCa(T, S)` |
| `omega = Ca * CO3 / Ksp` | `omega = (Ca_µmol_kg * 1e-6) * (CO3_µmol_kg * 1e-6) / KCa` |

---
> **References:**  
> Mucci (1983). *Am. J. Sci.* 283, 780–799.  
> Millero (2010). *Mar. Freshwater Res.* 61, 139–142.

## 📦 Step 1 — Install Dependencies

In [2]:
!pip install PyCO2SYS openpyxl --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.6/110.6 kB 8.6 MB/s eta 0:00:00


## 📚 Step 2 — Imports

In [3]:
import numpy as np
import pandas as pd
import PyCO2SYS as pyco2
import time

print(f"PyCO2SYS version : {pyco2.__version__}")
print(f"pandas  version  : {pd.__version__}")
print(f"numpy   version  : {np.__version__}")

PyCO2SYS version : 1.8.3.4
pandas  version  : 2.2.2
numpy   version  : 2.0.2


## ⚙️ Step 3 — Settings

Edit the values in **this cell** and the **Rock Composition cell** below to configure the full model run.

| Parameter | Valid options |
|---|---|
| `ROCK_NAME` | `"calcite"` \| `"dolomite"` \| `"basalt_flood"` \| `"custom"` |
| `DEGASS_CASE` | `"close"` (ALK+DIC, closed) \| `"degass"` (ALK+pCO₂, open) |
| `OPT_K_CARBONIC` | 1 Roy1993 \| 4 Mehrbach1973 \| 6 Millero2006 \| 10 Lueker2000 \| **14 Millero2010** |
| `OPT_PH_SCALE` | **1 Total** \| 2 Seawater \| 3 Free \| 4 NBS |

In [4]:
# ╔══════════════════════════════════════════════════════════════╗
# ║                 FEEDSTOCK / ROCK TYPE                       ║
# ╚══════════════════════════════════════════════════════════════╝
ROCK_NAME = "basalt_flood"   # ← CHANGE: calcite | dolomite | basalt_flood | custom

# ╔══════════════════════════════════════════════════════════════╗
# ║                   DEGASSING MODE                            ║
# ║  "close"  → closed system : ALK + DIC  (seacarb flag 15)   ║
# ║  "degass" → open system   : ALK + pCO₂ (seacarb flag 24)   ║
# ╚══════════════════════════════════════════════════════════════╝
DEGASS_CASE = "close"        # ← CHANGE: close | degass

# ╔══════════════════════════════════════════════════════════════╗
# ║               SUPERSATURATION THRESHOLD                     ║
# ╚══════════════════════════════════════════════════════════════╝
OMEGA_THRESH = 5.0           # Ω_calcite target (R model default: 5)
ERR_TOL      = 0.1           # Convergence tolerance |Ω − OMEGA_THRESH| < ERR_TOL

# ╔══════════════════════════════════════════════════════════════╗
# ║         PyCO2SYS EQUILIBRIUM CONSTANTS                      ║
# ║  Exactly as used in the PyCO2SYS Batch Excel notebook       ║
# ║  opt_k_carbonic:                                            ║
# ║    1  = Roy et al. (1993)                                   ║
# ║    4  = Mehrbach et al. (1973)                              ║
# ║    6  = Millero (2006)  ≈ seacarb k1k2="m06"               ║
# ║   10  = Lueker et al. (2000)                                ║
# ║   14  = Millero (2010) ← matches batch notebook (DEFAULT)   ║
# ╚══════════════════════════════════════════════════════════════╝
OPT_K_CARBONIC = 14          # ← CHANGE if needed
OPT_PH_SCALE   = 1           # 1=Total | 2=Seawater | 3=Free | 4=NBS
PRESSURE       = 0.0         # dbar — surface/lab (matches batch notebook)

# ╔══════════════════════════════════════════════════════════════╗
# ║          ITERATION CONTROL (mirrors R model defaults)       ║
# ║  ROCK_MAX    : max rock added, µmol/kg                      ║
# ║  NUM_J_STEPS : coarse outer loop  (R model: 1e4)            ║
# ║  NUM_K_STEPS : fine inner loop    (R model: 100)            ║
# ╚══════════════════════════════════════════════════════════════╝
ROCK_MAX    = 5000   # µmol/kg  (≈ R model rock_max=5 mol/L; adjust to your data range)
NUM_J_STEPS = 10000  # coarse steps
NUM_K_STEPS = 100    # fine refinement steps

# Scale factors — R model defaults (1.0 = all ions dissolve congruently)
SCALE_CA = SCALE_MG = SCALE_FE2 = SCALE_NA = SCALE_K = 1.0
SCALE_DIC = SCALE_ALK = 1.0

print("Settings loaded ✓")
print(f"  Rock type      : {ROCK_NAME}")
print(f"  Degassing      : {DEGASS_CASE}")
print(f"  Ω threshold    : {OMEGA_THRESH}  (err_tol = {ERR_TOL})")
print(f"  opt_k_carbonic : {OPT_K_CARBONIC}")
print(f"  opt_pH_scale   : {OPT_PH_SCALE}")
print(f"  Pressure       : {PRESSURE} dbar")
print(f"  Rock max       : {ROCK_MAX} µmol/kg")

Settings loaded ✓
  Rock type      : basalt_flood
  Degassing      : close
  Ω threshold    : 5.0  (err_tol = 0.1)
  opt_k_carbonic : 14
  opt_pH_scale   : 1
  Pressure       : 0.0 dbar
  Rock max       : 5000 µmol/kg


### 🪨 Rock Molar Composition

Molar fractions of cations per mole of rock dissolved — **identical to the R model**.  
To use a custom feedstock, set `ROCK_NAME = "custom"` above and fill in the values in the `elif` block.

In [5]:
if ROCK_NAME == "calcite":
    CA_MOL  = 1.0;  MG_MOL  = 0.0;  NA_MOL = 0.0; K_MOL = 0.0; FE2_MOL = 0.0

elif ROCK_NAME == "dolomite":
    CA_MOL  = 0.5;  MG_MOL  = 0.5;  NA_MOL = 0.0; K_MOL = 0.0; FE2_MOL = 0.0

elif ROCK_NAME == "basalt_flood":
    CA_MOL  = 0.22; MG_MOL  = 0.22; NA_MOL = 0.11; K_MOL = 0.01; FE2_MOL = 0.14

elif ROCK_NAME == "custom":
    # ── Edit your molar fractions here ───────────────────────────────────
    CA_MOL  = 0.22   # Ca   mole fraction
    MG_MOL  = 0.22   # Mg   mole fraction
    NA_MOL  = 0.11   # Na   mole fraction
    K_MOL   = 0.01   # K    mole fraction
    FE2_MOL = 0.14   # Fe²⁺ mole fraction
    # ─────────────────────────────────────────────────────────────────────

else:
    raise ValueError(f"Unknown ROCK_NAME='{ROCK_NAME}'. Use: calcite | dolomite | basalt_flood | custom")

# Alkalinity released per mole of rock (R model: rock_alk_ratio)
# monovalent (Na, K) → +1 charge each; divalent (Fe²⁺, Mg, Ca) → +2 each
ROCK_ALK_RATIO = NA_MOL + K_MOL + (FE2_MOL + MG_MOL + CA_MOL) * 2

# Step sizes
ADD_SINGLE_J = ROCK_MAX / NUM_J_STEPS      # µmol/kg per coarse step
ADD_SINGLE_K = ADD_SINGLE_J / NUM_K_STEPS  # µmol/kg per fine step

print(f"Rock composition for '{ROCK_NAME}':")
print(f"  Ca   = {CA_MOL:.4f} mol/mol-rock")
print(f"  Mg   = {MG_MOL:.4f}")
print(f"  Na   = {NA_MOL:.4f}")
print(f"  K    = {K_MOL:.4f}")
print(f"  Fe²⁺ = {FE2_MOL:.4f}")
print(f"  rock_alk_ratio = {ROCK_ALK_RATIO:.4f} mol-alk / mol-rock")
print(f"  ADD_SINGLE_J = {ADD_SINGLE_J:.6f} µmol/kg (coarse step)")
print(f"  ADD_SINGLE_K = {ADD_SINGLE_K:.8f} µmol/kg (fine step)")

Rock composition for 'basalt_flood':
  Ca   = 0.2200 mol/mol-rock
  Mg   = 0.2200
  Na   = 0.1100
  K    = 0.0100
  Fe²⁺ = 0.1400
  rock_alk_ratio = 1.2800 mol-alk / mol-rock
  ADD_SINGLE_J = 0.500000 µmol/kg (coarse step)
  ADD_SINGLE_K = 0.00500000 µmol/kg (fine step)


## 📂 Step 4 — Load Raw Discharge Data

Upload your Excel file — **same format as the PyCO2SYS Batch Excel notebook input**.

**Required columns:** `pH` · `Temp (°C)` · `Ca (µmol/l)` · `Total Alkalinity (µmol/l)` · `Salinity (PSU)`

> The initial PyCO2SYS run (Step 6) derives all quantities needed by the ERW loop  
> (DIC, CO₃²⁻, HCO₃⁻, pCO₂, Ω_calcite …) directly from these five columns.

In [8]:
# ── File path ─────────────────────────────────────────────────────────────
EXCEL_PATH = "Untitled_spreadsheet-11.xlsx"   # ← update to your filename
SHEET_NAME = 0                                 # 0 = first sheet

# ── Column names (edit if your headers differ) ────────────────────────────
COL_PH   = "pH"
COL_TEMP = "Temp (°C)"
COL_CA   = "Ca (µmol/l)"
COL_TA   = "Total Alkalinity (µmol/l)"
COL_SAL  = "Salinity (PSU)"

SAL_WARN_THRESHOLD = 0.001   # Mucci 1983 Ksp calibration range flag

# ── Load ──────────────────────────────────────────────────────────────────
df_raw = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_NAME)
print(f"Raw shape  : {df_raw.shape}")
print(f"Columns    : {list(df_raw.columns)}")

NEEDED_COLS = [COL_PH, COL_TEMP, COL_CA, COL_TA, COL_SAL]
df = df_raw[NEEDED_COLS].copy()
n_before = len(df)
df = df.dropna().reset_index(drop=True)
print(f"\nComplete rows : {len(df)}  ({n_before - len(df)} dropped for missing values)")
print()
print(df.head(5).to_string())

Raw shape  : (1500, 6)
Columns    : ['pH', 'Temp (°C)', 'Ca (µmol/l)', 'Total Alkalinity (µmol/l)', 'Salinity (PSU)', 'Discharge (m³/s)']

Complete rows : 1500  (0 dropped for missing values)

     pH  Temp (°C)  Ca (µmol/l)  Total Alkalinity (µmol/l)  Salinity (PSU)
0  5.20      24.85         38.0                      190.0        0.004035
1  5.70      24.85         43.0                      170.0        0.004740
2  5.75      -4.43         60.0                      359.0        0.001345
3  5.80      24.85         54.0                      180.0        0.004740
4  5.90      27.85         45.0                      210.0        0.003843


## 🔬 Step 5 — Mucci (1983) Ksp Functions

**Identical to the PyCO2SYS Batch Excel notebook.**  
Passed explicitly to `PyCO2SYS` via `k_calcite` / `k_aragonite` and used inside the ERW loop to compute  
Ω = [Ca²⁺][CO₃²⁻] / K_sp, replacing `seacarb::Kspc()`.

$$\log K_{sp}^{\mathrm{cal}} = -171.9065 - 0.077993\,T_K + \frac{2839.319}{T_K} + 71.595\,\log_{10}(T_K) + \left(-0.77712 + 0.0028426\,T_K + \frac{178.34}{T_K}\right)\sqrt{S} - 0.07711\,S + 0.0041249\,S^{3/2}$$

In [9]:
def mucci1983_KCa(TempC, Sal):
    """Calcite Ksp at 1 atm — Mucci (1983). Works on scalars or numpy arrays."""
    TK  = np.asarray(TempC, dtype=float) + 273.15
    S   = np.asarray(Sal,   dtype=float)
    logK = (-171.9065
             - 0.077993 * TK
             + 2839.319 / TK
             + 71.595   * np.log10(TK)
             + (-0.77712 + 0.0028426 * TK + 178.34 / TK) * np.sqrt(S)
             - 0.07711  * S
             + 0.0041249 * S**1.5)
    return 10.0**logK


def mucci1983_KAr(TempC, Sal):
    """Aragonite Ksp at 1 atm — Mucci (1983). Works on scalars or numpy arrays."""
    TK  = np.asarray(TempC, dtype=float) + 273.15
    S   = np.asarray(Sal,   dtype=float)
    logK = (-171.945
             - 0.077993 * TK
             + 2903.293 / TK
             + 71.595   * np.log10(TK)
             + (-0.068393 + 0.0017276 * TK + 88.135 / TK) * np.sqrt(S)
             - 0.10018  * S
             + 0.0059415 * S**1.5)
    return 10.0**logK


# Sanity check — same reference values as batch notebook
print("Mucci 1983 reference check (T=25°C, S=35):")
print(f"  KCa = {mucci1983_KCa(25,35):.4e}  (expect ~3.36e-7 mol²/kg²)")
print(f"  KAr = {mucci1983_KAr(25,35):.4e}  (expect ~4.57e-7 mol²/kg²)")
print(f"  KAr/KCa = {mucci1983_KAr(25,35)/mucci1983_KCa(25,35):.4f}  (expect ~1.36)")
print("Functions defined ✓")

Mucci 1983 reference check (T=25°C, S=35):
  KCa = 4.2724e-07  (expect ~3.36e-7 mol²/kg²)
  KAr = 6.4818e-07  (expect ~4.57e-7 mol²/kg²)
  KAr/KCa = 1.5171  (expect ~1.36)
Functions defined ✓


## 🚀 Step 6 — Initial PyCO2SYS Run (Discharge Baseline)

**Vectorised — exactly as in the PyCO2SYS Batch Excel notebook:**
- `par1 = Total Alkalinity` · `par1_type = 1`
- `par2 = pH`               · `par2_type = 3`
- `opt_k_carbonic = OPT_K_CARBONIC`
- Mucci (1983) Ksp via `k_calcite` / `k_aragonite`
- Measured [Ca²⁺] via `total_calcium`

The outputs become the **initial state** for the ERW loop.

In [10]:
# ── Extract input arrays ──────────────────────────────────────────────────
pH_arr  = df[COL_PH  ].to_numpy(dtype=float)
T_arr   = df[COL_TEMP].to_numpy(dtype=float)
Ca_arr  = df[COL_CA  ].to_numpy(dtype=float)
TA_arr  = df[COL_TA  ].to_numpy(dtype=float)
S_arr   = df[COL_SAL ].to_numpy(dtype=float)

# Mucci 1983 Ksp arrays for baseline
KCa_arr = mucci1983_KCa(T_arr, S_arr)
KAr_arr = mucci1983_KAr(T_arr, S_arr)

print(f"Running initial PyCO2SYS on {len(df)} rows (vectorised) ...")

# ── Single vectorised call — identical to PyCO2SYS Batch notebook ─────────
res_init = pyco2.sys(
    par1            = TA_arr,
    par1_type       = 1,            # Total Alkalinity
    par2            = pH_arr,
    par2_type       = 3,            # pH
    temperature     = T_arr,
    salinity        = S_arr,
    pressure        = PRESSURE,
    opt_k_carbonic  = OPT_K_CARBONIC,
    opt_pH_scale    = OPT_PH_SCALE,
    total_calcium   = Ca_arr,      # measured [Ca²⁺] µmol/kg — bypasses salinity estimate
    k_calcite       = KCa_arr,     # Mucci 1983 KCa — passed explicitly
    k_calcite_out   = KCa_arr,
    k_aragonite     = KAr_arr,     # Mucci 1983 KAr — passed explicitly
    k_aragonite_out = KAr_arr,
)

def _arr(key):
    return np.atleast_1d(res_init[key]).ravel().astype(float)

# Manual Omega verification (as in batch notebook)
CO3_mol = _arr('carbonate')     * 1e-6
Ca_mol  = _arr('total_calcium') * 1e-6
Omega_Ca_manual = (CO3_mol * Ca_mol) / KCa_arr
Omega_Ca_pyco2  = _arr('saturation_calcite')
max_delta = np.abs(Omega_Ca_manual - Omega_Ca_pyco2).max()

# ── Assemble baseline DataFrame ───────────────────────────────────────────
df_init = pd.DataFrame({
    "pH_input"          : pH_arr,
    "Temp_C"            : T_arr,
    "Salinity_PSU"      : S_arr,
    "TA_umol_kg"        : TA_arr,
    "Ca_umol_kg"        : Ca_arr,
    "KCa_M83"           : KCa_arr,
    "KAr_M83"           : KAr_arr,
    # Carbonate system
    "DIC_umol_kg"       : _arr('dic'),
    "CO2aq_umol_kg"     : _arr('aqueous_CO2'),
    "HCO3_umol_kg"      : _arr('bicarbonate'),
    "CO3_umol_kg"       : _arr('carbonate'),
    "pCO2_uatm"         : _arr('pCO2'),
    "fCO2_uatm"         : _arr('fCO2'),
    # pH on all scales
    "pH_Total"          : _arr('pH_total'),
    "pH_Seawater"       : _arr('pH_sws'),
    "pH_Free"           : _arr('pH_free'),
    "pH_NBS"            : _arr('pH_nbs'),
    # Saturation states
    "Omega_calcite"     : Omega_Ca_pyco2,
    "Omega_aragonite"   : _arr('saturation_aragonite'),
    "Omega_Ca_manual"   : Omega_Ca_manual,
    "Delta_Omega_Ca"    : np.abs(Omega_Ca_manual - Omega_Ca_pyco2),
    # Equilibrium constants
    "K1"                : _arr('k_carbonic_1'),
    "K2"                : _arr('k_carbonic_2'),
    "pK1"               : -np.log10(_arr('k_carbonic_1')),
    "pK2"               : -np.log10(_arr('k_carbonic_2')),
    # Totals and buffer
    "Total_Borate"      : _arr('total_borate'),
    "Total_Sulfate"     : _arr('total_sulfate'),
    "OH_umol_kg"        : _arr('hydroxide'),
    "Revelle_factor"    : _arr('revelle_factor'),
    # QA
    "QA_Sal_flag"       : np.where(S_arr < SAL_WARN_THRESHOLD,
                                    "WARNING: S<0.001 (Ksp extrapolated)", "OK"),
})

print("\n✓ Initial PyCO2SYS done.")
print(f"  Rows processed       : {len(df_init)}")
print(f"  Omega_calcite range  : [{df_init['Omega_calcite'].min():.3f}, {df_init['Omega_calcite'].max():.3f}]")
print(f"  Already >= {OMEGA_THRESH}     : {(df_init['Omega_calcite'] >= OMEGA_THRESH).sum()} rows")
print(f"  Max |Ω manual-pyco2| : {max_delta:.2e}  (should be ~0)")
print(f"  Low salinity rows    : {(S_arr < SAL_WARN_THRESHOLD).sum()}")
print()
df_init[['pH_input','Temp_C','Salinity_PSU','TA_umol_kg','Ca_umol_kg',
         'DIC_umol_kg','CO3_umol_kg','pCO2_uatm','Omega_calcite']].head(5)

Running initial PyCO2SYS on 1500 rows (vectorised) ...

✓ Initial PyCO2SYS done.
  Rows processed       : 1500
  Omega_calcite range  : [0.000, 189.151]
  Already >= 5.0     : 182 rows
  Max |Ω manual-pyco2| : 2.84e-14  (should be ~0)
  Low salinity rows    : 180



,pH_input,Temp_C,Salinity_PSU,TA_umol_kg,Ca_umol_kg,DIC_umol_kg,CO3_umol_kg,pCO2_uatm,Omega_calcite
0,5.20,24.85,0.004035,190.0,38.0,2924.023767,0.001560,80019.098592,0.000016
1,5.70,24.85,0.004740,170.0,43.0,926.320717,0.004347,22128.914782,0.000051
2,5.75,-4.43,0.001345,359.0,60.0,3128.502280,0.004194,29878.344630,0.000056
3,5.80,24.85,0.004740,180.0,54.0,814.158142,0.005778,18557.314689,0.000085
4,5.90,27.85,0.003843,210.0,45.0,776.537464,0.008918,17936.884699,0.000114


## 🔄 Step 7 — ERW / CDR Iterative Loop

**Direct Python port of the R model loop.** For each sample `i`:

- **If Ω_calcite ≥ `OMEGA_THRESH`** → already supersaturated, record initial state and continue.
- **Otherwise** iterate mineral addition:
  - **Outer j-loop (coarse):** add `ADD_SINGLE_J` µmol/kg of rock per step (up to `ROCK_MAX`).
  - **Inner k-loop (fine):** back off one j-step, then refine until within `ERR_TOL`.
  - If Ω is decreasing → flag as `cannot_reach` (success_flag = 2) and break.

> **Unit note:** concentrations are in µmol/kg throughout.  
> Ω = (Ca [µmol/kg] × 10⁻⁶) × (CO₃²⁻ [µmol/kg] × 10⁻⁶) / K_sp [mol²/kg²]  
> Salinity update: Δsal [g/kg] = Σ (ion_add [µmol/kg] × MW [g/mol]) × 10⁻⁶

In [11]:
num_sample = len(df_init)

# ── Pre-allocate output arrays (R model naming convention) ────────────────
v_j_steps      = np.zeros(num_sample, dtype=int)
v_k_steps      = np.zeros(num_sample, dtype=int)
v_add          = np.zeros(num_sample)            # total rock added (µmol/kg)
v_omega_flag   = np.zeros(num_sample, dtype=int) # 1 = reached OMEGA_THRESH
v_success_flag = np.zeros(num_sample, dtype=int) # 1=converged | 2=cant reach | 0=NaN err
v_seacarb_flag = np.ones(num_sample,  dtype=int) # 0 = PyCO2SYS returned NaN

_nan = np.full(num_sample, np.nan)
v_salinity_final = _nan.copy()
v_omega_final    = _nan.copy()
v_ca_final       = _nan.copy()
v_co3_final      = _nan.copy()
v_alk_final      = _nan.copy()
v_dic_final      = _nan.copy()
v_hco3_final     = _nan.copy()
v_ph_final       = _nan.copy()
v_co2_final      = _nan.copy()
v_pco2_final     = _nan.copy()
v_fco2_final     = _nan.copy()

print(f"Pre-allocated {num_sample} output arrays ✓")

Pre-allocated 1500 output arrays ✓


In [12]:
# Helper: scalar extractor (kept for compatibility with cards cell)
def _scalar(res, key):
    return float(np.atleast_1d(res[key]).ravel()[0])

# PAR2 type: 2=DIC (closed) | 4=pCO2 (open)
PAR2_TYPE = 2 if DEGASS_CASE == "close" else 4

# Salinity coefficient: PSU gained per µmol/kg of rock added (linear)
SAL_COEFF = (
    CA_MOL * SCALE_CA * 40 + ROCK_ALK_RATIO * SCALE_DIC * 61 +
    MG_MOL * SCALE_MG * 24 + FE2_MOL * SCALE_FE2 * 56 +
    NA_MOL * SCALE_NA * 23 + K_MOL * SCALE_K * 39
) * 1e-6

print(f"PAR2_TYPE = {PAR2_TYPE} → {'DIC' if PAR2_TYPE==2 else 'pCO2'}")
print(f"SAL_COEFF = {SAL_COEFF:.6e} PSU per µmol/kg rock")
print("Helpers ready ✓")

PAR2_TYPE = 2 → DIC
SAL_COEFF = 1.029200e-04 PSU per µmol/kg rock
Helpers ready ✓


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  VECTORISED ERW LOOP
#
#  Speedup vs row-by-row: instead of one pyco2.sys() call per row per
#  j-step, we batch ALL still-active rows into ONE call at each j-step.
#  For 1 500 rows × 10 000 j-steps that shrinks ~15 M calls → ~10 K calls.
# ══════════════════════════════════════════════════════════════════════════

t0 = time.time()

# ── Baseline arrays (immutable – additions are cumulative from t=0) ────
T0     = df_init['Temp_C'].values
Ca0    = df_init['Ca_umol_kg'].values
DIC0   = df_init['DIC_umol_kg'].values
TA0    = df_init['TA_umol_kg'].values
S0     = df_init['Salinity_PSU'].values
pCO2_0 = df_init['pCO2_uatm'].values
omega0 = df_init['Omega_calcite'].values

# ── Already-supersaturated rows: copy baseline directly ────────────────
ss = np.where(omega0 >= OMEGA_THRESH)[0]
v_omega_flag[ss] = v_success_flag[ss] = 1
v_salinity_final[ss] = S0[ss]
v_omega_final[ss]    = omega0[ss]
v_ca_final[ss]       = Ca0[ss]
v_co3_final[ss]      = df_init['CO3_umol_kg'].values[ss]
v_hco3_final[ss]     = df_init['HCO3_umol_kg'].values[ss]
v_alk_final[ss]      = TA0[ss]
v_dic_final[ss]      = DIC0[ss]
v_ph_final[ss]       = df_init['pH_Total'].values[ss]
v_co2_final[ss]      = df_init['CO2aq_umol_kg'].values[ss]
v_pco2_final[ss]     = pCO2_0[ss]
v_fco2_final[ss]     = df_init['fCO2_uatm'].values[ss]

# ── NaN baseline rows ──────────────────────────────────────────────────
v_seacarb_flag[np.isnan(omega0)] = 0

# ── Active set: rows still needing rock addition ───────────────────────
active = np.where(~(omega0 >= OMEGA_THRESH) & ~np.isnan(omega0))[0].copy()

omega_prev = omega0.copy()           # per-row omega from previous j-step
needs_k      = np.zeros(num_sample, bool)
pre_cross_j  = np.zeros(num_sample, int)

# ── Helper: write batch results into output arrays ─────────────────────
OUT_KEYS = [
    (v_hco3_final,  'bicarbonate'),
    (v_alk_final,   'alkalinity'),
    (v_dic_final,   'dic'),
    (v_ph_final,    'pH_total'),
    (v_co2_final,   'aqueous_CO2'),
    (v_pco2_final,  'pCO2'),
    (v_fco2_final,  'fCO2'),
]

def store_batch(rows, pos_in_batch, res, ca_b, co3_b, sal_b, omega_b, rock_b, j_val, k_val=0):
    """Write converged results for a subset of rows from a batch result."""
    v_omega_final[rows]    = omega_b[pos_in_batch]
    v_ca_final[rows]       = ca_b[pos_in_batch]
    v_co3_final[rows]      = co3_b[pos_in_batch]
    v_salinity_final[rows] = sal_b[pos_in_batch]
    for arr, key in OUT_KEYS:
        arr[rows] = np.atleast_1d(res[key]).ravel()[pos_in_batch]
    v_j_steps[rows] = j_val
    v_k_steps[rows] = k_val
    if np.isscalar(rock_b):
        v_add[rows] = rock_b
    else:
        v_add[rows] = rock_b[pos_in_batch]

# ══════════════════════════════════════════════════════════════════════════
#  PHASE 1 — Coarse j-loop (vectorised over active rows)
# ══════════════════════════════════════════════════════════════════════════
print(f"Starting vectorised j-loop on {len(active)} rows …")

for j in range(1, NUM_J_STEPS + 1):
    if len(active) == 0:
        break

    rock_j = ADD_SINGLE_J * j   # cumulative rock added (scalar)

    # Compute updates for ALL active rows simultaneously (linear in rock_j)
    ca_b   = Ca0[active]  + rock_j * CA_MOL  * SCALE_CA
    dic_b  = DIC0[active] + rock_j * ROCK_ALK_RATIO * SCALE_DIC
    alk_b  = TA0[active]  + rock_j * ROCK_ALK_RATIO * SCALE_ALK
    sal_b  = S0[active]   + rock_j * SAL_COEFF
    par2_b = dic_b if DEGASS_CASE == "close" else pCO2_0[active]
    kca_b  = mucci1983_KCa(T0[active], sal_b)
    kar_b  = mucci1983_KAr(T0[active], sal_b)

    # ── ONE vectorised pyco2.sys call for all active rows ─────────────
    res = pyco2.sys(
        par1=alk_b,  par1_type=1,
        par2=par2_b, par2_type=PAR2_TYPE,
        temperature=T0[active], salinity=sal_b, pressure=PRESSURE,
        opt_k_carbonic=OPT_K_CARBONIC, opt_pH_scale=OPT_PH_SCALE,
        total_calcium=ca_b,
        k_calcite=kca_b,     k_calcite_out=kca_b,
        k_aragonite=kar_b,   k_aragonite_out=kar_b,
    )

    co3_b   = np.atleast_1d(res['carbonate']).ravel().astype(float)
    omega_b = (ca_b * 1e-6) * (co3_b * 1e-6) / kca_b

    # NaN errors
    nan_b = np.isnan(co3_b)
    v_seacarb_flag[active[nan_b]] = 0

    # Omega decrease → cannot reach threshold
    if j > 1:
        dec_b = (~nan_b) & (omega_b < omega_prev[active])
        v_success_flag[active[dec_b]] = 2
    else:
        dec_b = np.zeros(len(active), bool)

    # Update per-row previous omega (only for valid, non-decreasing rows)
    valid_b = ~nan_b & ~dec_b
    omega_prev[active[valid_b]] = omega_b[valid_b]

    # Threshold crossing
    crossed_b = valid_b & (omega_b >= OMEGA_THRESH)
    v_omega_flag[active[crossed_b]] = 1

    if crossed_b.any():
        cross_rows = active[crossed_b]
        cross_pos  = np.where(crossed_b)[0]          # positions in batch
        omega_c    = omega_b[cross_pos]
        tol_ok     = (omega_c - OMEGA_THRESH) < ERR_TOL

        # Direct convergence (within tolerance in j-step)
        d_rows = cross_rows[tol_ok]
        d_pos  = cross_pos[tol_ok]
        if len(d_rows):
            v_success_flag[d_rows] = 1
            store_batch(d_rows, d_pos, res, ca_b, co3_b, sal_b, omega_b, rock_j, j)

        # Overshot tolerance → queue for k-refinement
        r_rows = cross_rows[~tol_ok]
        if len(r_rows):
            needs_k[r_rows]     = True
            pre_cross_j[r_rows] = j

    # Remove from active: NaN / decrease / crossed
    active = active[valid_b & ~crossed_b]

    if j % 500 == 0:
        sf = v_success_flag.tolist()
        print(f"  j={j:>6}  active={len(active):>5}  "
              f"converged={sf.count(1)}  cant_reach={sf.count(2)}  "
              f"elapsed={time.time()-t0:.1f}s")

# ══════════════════════════════════════════════════════════════════════════
#  PHASE 2 — Fine k-loop (vectorised over rows needing refinement)
# ══════════════════════════════════════════════════════════════════════════
k_rows = np.where(needs_k)[0]
print(f"\nK-refinement phase: {len(k_rows)} rows")

if len(k_rows):
    rock_base = ADD_SINGLE_J * (pre_cross_j[k_rows] - 1)  # per-row, vector!
    j_base    = pre_cross_j[k_rows] - 1                    # j before crossing
    k_active  = np.arange(len(k_rows))                     # indices into k_rows

    for k in range(1, NUM_K_STEPS + 1):
        if len(k_active) == 0:
            break

        kr       = k_rows[k_active]
        rock_jk  = rock_base[k_active] + ADD_SINGLE_K * k  # per-row

        ca_b   = Ca0[kr]  + rock_jk * CA_MOL  * SCALE_CA
        dic_b  = DIC0[kr] + rock_jk * ROCK_ALK_RATIO * SCALE_DIC
        alk_b  = TA0[kr]  + rock_jk * ROCK_ALK_RATIO * SCALE_ALK
        sal_b  = S0[kr]   + rock_jk * SAL_COEFF
        par2_b = dic_b if DEGASS_CASE == "close" else pCO2_0[kr]
        kca_b  = mucci1983_KCa(T0[kr], sal_b)
        kar_b  = mucci1983_KAr(T0[kr], sal_b)

        res = pyco2.sys(
            par1=alk_b,  par1_type=1,
            par2=par2_b, par2_type=PAR2_TYPE,
            temperature=T0[kr], salinity=sal_b, pressure=PRESSURE,
            opt_k_carbonic=OPT_K_CARBONIC, opt_pH_scale=OPT_PH_SCALE,
            total_calcium=ca_b,
            k_calcite=kca_b,   k_calcite_out=kca_b,
            k_aragonite=kar_b, k_aragonite_out=kar_b,
        )

        co3_b   = np.atleast_1d(res['carbonate']).ravel().astype(float)
        omega_b = (ca_b * 1e-6) * (co3_b * 1e-6) / kca_b

        nan_b     = np.isnan(co3_b)
        v_seacarb_flag[kr[nan_b]] = 0

        crossed_b = (~nan_b) & (omega_b >= OMEGA_THRESH)
        if crossed_b.any():
            cross_pos  = np.where(crossed_b)[0]
            cross_rows = kr[crossed_b]
            omega_c    = omega_b[cross_pos]
            tol_ok     = (omega_c - OMEGA_THRESH) < ERR_TOL
            v_success_flag[cross_rows[tol_ok]] = 1
            v_omega_flag[cross_rows]            = 1
            store_batch(
                cross_rows, cross_pos, res,
                ca_b, co3_b, sal_b, omega_b,
                rock_jk, j_base[k_active[crossed_b]], k
            )

        k_active = k_active[~nan_b & ~crossed_b]

# ── Final report ──────────────────────────────────────────────────────────
total_time = time.time() - t0
sf = v_success_flag.tolist()
print(f"\n✓ Vectorised ERW loop complete — total time: {total_time:.1f}s")
print(f"  converged={sf.count(1)}  cant_reach={sf.count(2)}  NaN_err={sf.count(0)}")

Starting vectorised j-loop on 1318 rows …
  j=   500  active= 1236  converged=201  cant_reach=63  elapsed=148.7s


## 📊 Step 8 — Build Results DataFrame

Combines initial PyCO2SYS baseline + ERW loop outputs + Δ-change columns + flags.

In [ ]:
flag_desc = {0: "NaN_error", 1: "converged", 2: "cannot_reach_omega"}

# Delta columns (ERW-induced change)
delta_alk  = v_alk_final  - df_init['TA_umol_kg'].values
delta_dic  = v_dic_final  - df_init['DIC_umol_kg'].values
delta_ca   = v_ca_final   - df_init['Ca_umol_kg'].values
delta_ph   = v_ph_final   - df_init['pH_Total'].values
delta_co3  = v_co3_final  - df_init['CO3_umol_kg'].values
delta_pco2 = v_pco2_final - df_init['pCO2_uatm'].values

df_out = pd.DataFrame({
    # ── Discharge baseline (initial PyCO2SYS outputs) ─────────────────────
    "pH_init"            : df_init['pH_input'].values,
    "Temp_C"             : df_init['Temp_C'].values,
    "Sal_init"           : df_init['Salinity_PSU'].values,
    "TA_init_umol_kg"    : df_init['TA_umol_kg'].values,
    "Ca_init_umol_kg"    : df_init['Ca_umol_kg'].values,
    "DIC_init_umol_kg"   : df_init['DIC_umol_kg'].values,
    "CO3_init_umol_kg"   : df_init['CO3_umol_kg'].values,
    "HCO3_init_umol_kg"  : df_init['HCO3_umol_kg'].values,
    "pCO2_init_uatm"     : df_init['pCO2_uatm'].values,
    "fCO2_init_uatm"     : df_init['fCO2_uatm'].values,
    "CO2aq_init_umol_kg" : df_init['CO2aq_umol_kg'].values,
    "Omega_calcite_init" : df_init['Omega_calcite'].values,
    "Omega_arag_init"    : df_init['Omega_aragonite'].values,
    "pH_Total_init"      : df_init['pH_Total'].values,
    "pH_Seawater_init"   : df_init['pH_Seawater'].values,
    "pH_Free_init"       : df_init['pH_Free'].values,
    "pH_NBS_init"        : df_init['pH_NBS'].values,
    "K1_init"            : df_init['K1'].values,
    "K2_init"            : df_init['K2'].values,
    "KCa_M83_init"       : df_init['KCa_M83'].values,
    "KAr_M83_init"       : df_init['KAr_M83'].values,
    "Revelle_init"       : df_init['Revelle_factor'].values,
    # ── ERW loop outputs (post-rock-addition state) ───────────────────────
    "rock_add_umol_kg"   : v_add,
    "j_steps"            : v_j_steps,
    "k_steps"            : v_k_steps,
    "Sal_final"          : v_salinity_final,
    "Omega_calcite_final": v_omega_final,
    "Ca_final_umol_kg"   : v_ca_final,
    "CO3_final_umol_kg"  : v_co3_final,
    "HCO3_final_umol_kg" : v_hco3_final,
    "TA_final_umol_kg"   : v_alk_final,
    "DIC_final_umol_kg"  : v_dic_final,
    "pH_final"           : v_ph_final,
    "CO2aq_final_umol_kg": v_co2_final,
    "pCO2_final_uatm"    : v_pco2_final,
    "fCO2_final_uatm"    : v_fco2_final,
    # ── Delta (ERW-induced change) ────────────────────────────────────────
    "Delta_TA_umol_kg"   : delta_alk,
    "Delta_DIC_umol_kg"  : delta_dic,
    "Delta_Ca_umol_kg"   : delta_ca,
    "Delta_pH"           : delta_ph,
    "Delta_CO3_umol_kg"  : delta_co3,
    "Delta_pCO2_uatm"    : delta_pco2,
    # ── Flags (R model convention) ────────────────────────────────────────
    "omega_flag"         : v_omega_flag,    # 1 = reached OMEGA_THRESH
    "success_flag"       : v_success_flag,  # 1=OK | 2=cant_reach | 0=NaN
    "pyco2_flag"         : v_seacarb_flag,  # 0 = PyCO2SYS error
    "success_label"      : [flag_desc.get(f, "unknown") for f in v_success_flag],
    "QA_Sal_flag"        : df_init['QA_Sal_flag'].values,
})

print(f"Results DataFrame: {df_out.shape[0]} rows × {df_out.shape[1]} columns")
df_out.head(5)

## 🔍 Step 9 — Run Summary & QA

In [ ]:
n_converged  = int((v_success_flag == 1).sum())
n_cant_reach = int((v_success_flag == 2).sum())
n_nan_err    = int((v_seacarb_flag == 0).sum())
n_already_ss = int(((v_success_flag == 1) & (v_add == 0)).sum())
n_low_sal    = int((df_init['Salinity_PSU'].values < SAL_WARN_THRESHOLD).sum())
mean_add     = float(v_add[v_add > 0].mean()) if (v_add > 0).any() else 0.0

print("═" * 60)
print("  ERW / CDR RUN SUMMARY")
print("═" * 60)
print(f"  Rock            : {ROCK_NAME}")
print(f"  Degassing       : {DEGASS_CASE}")
print(f"  Ω threshold     : {OMEGA_THRESH}  (err_tol={ERR_TOL})")
print(f"  opt_k_carbonic  : {OPT_K_CARBONIC}")
print("-" * 60)
print(f"  Rows processed  : {num_sample}")
print(f"  ✓ Converged     : {n_converged}  (success_flag = 1)")
print(f"  ✓ Already ≥ Ω   : {n_already_ss}  (no rock needed)")
print(f"  ✗ Cannot reach  : {n_cant_reach}  (success_flag = 2)")
print(f"  ✗ PyCO2SYS NaN  : {n_nan_err}  (pyco2_flag = 0)")
print(f"  ⚠ Low salinity  : {n_low_sal}  (S < {SAL_WARN_THRESHOLD})")
print("-" * 60)
print(f"  Mean rock added : {mean_add:.2f} µmol/kg  (rows where > 0)")
print(f"  Max rock added  : {v_add.max():.2f} µmol/kg")
print(f"  Ω_final range   : [{float(np.nanmin(v_omega_final)):.3f}, {float(np.nanmax(v_omega_final)):.3f}]")
print("═" * 60)

## 📈 Step 10 — Descriptive Statistics

In [ ]:
STAT_COLS = [
    'Omega_calcite_init', 'Omega_calcite_final',
    'rock_add_umol_kg',
    'TA_init_umol_kg', 'TA_final_umol_kg', 'Delta_TA_umol_kg',
    'DIC_init_umol_kg', 'DIC_final_umol_kg', 'Delta_DIC_umol_kg',
    'pCO2_init_uatm', 'pCO2_final_uatm', 'Delta_pCO2_uatm',
    'pH_Total_init', 'pH_final', 'Delta_pH',
]
pd.set_option('display.float_format', lambda x: f'{x:.4g}')
pd.set_option('display.max_columns', 16)
print("Descriptive statistics for key output variables:")
df_out[STAT_COLS].describe().round(4)

## 📋 Step 11 — Preview Key Results

In [ ]:
KEY_COLS = [
    'Temp_C', 'Sal_init',
    'TA_init_umol_kg', 'TA_final_umol_kg', 'Delta_TA_umol_kg',
    'Omega_calcite_init', 'Omega_calcite_final',
    'rock_add_umol_kg',
    'pH_Total_init', 'pH_final', 'Delta_pH',
    'pCO2_init_uatm', 'pCO2_final_uatm',
    'success_label', 'QA_Sal_flag',
]
pd.set_option('display.max_rows', 25)
print("First 20 rows — key outputs:")
df_out[KEY_COLS].head(20)

## 💾 Step 12 — Export Results to Excel

In [ ]:
OUTPUT_FILE = f"ERW_CDR_PyCO2SYS_{ROCK_NAME}_{DEGASS_CASE}.xlsx"

KEY_OUT_COLS = [
    'Temp_C', 'Sal_init', 'pH_init',
    'TA_init_umol_kg', 'Ca_init_umol_kg', 'DIC_init_umol_kg',
    'Omega_calcite_init', 'Omega_arag_init',
    'rock_add_umol_kg', 'j_steps', 'k_steps',
    'Sal_final', 'Omega_calcite_final',
    'TA_final_umol_kg', 'DIC_final_umol_kg', 'Ca_final_umol_kg',
    'CO3_final_umol_kg', 'HCO3_final_umol_kg',
    'pH_final', 'pCO2_final_uatm', 'fCO2_final_uatm',
    'Delta_TA_umol_kg', 'Delta_DIC_umol_kg', 'Delta_pH', 'Delta_pCO2_uatm',
    'success_label', 'pyco2_flag', 'QA_Sal_flag',
]

df_cant = df_out[df_out['success_flag'] == 2].copy()
df_lowS = df_out[df_out['QA_Sal_flag'] != 'OK'].copy()

with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    df_out.to_excel(writer,               sheet_name='All_Results',        index=False)
    df_out[KEY_OUT_COLS].to_excel(writer, sheet_name='Key_Outputs',        index=False)
    df_init.to_excel(writer,              sheet_name='Baseline_PyCO2SYS',  index=False)
    df_cant.to_excel(writer,              sheet_name='Cannot_Reach_Omega', index=False)
    df_lowS.to_excel(writer,              sheet_name='Flagged_LowSal',     index=False)
    df_out[STAT_COLS].describe().round(4).to_excel(writer, sheet_name='Statistics')

print(f"✓ Results written to: {OUTPUT_FILE}")
print(f"   All_Results        : {len(df_out)} rows × {len(df_out.columns)} cols")
print(f"   Key_Outputs        : {len(df_out)} rows × {len(KEY_OUT_COLS)} cols")
print(f"   Baseline_PyCO2SYS  : {len(df_init)} rows × {len(df_init.columns)} cols")
print(f"   Cannot_Reach_Omega : {len(df_cant)} rows")
print(f"   Flagged_LowSal     : {len(df_lowS)} rows")
print(f"   Statistics         : descriptive stats for {len(STAT_COLS)} variables")

## 📝 Step 13 — Per-Row Summary Cards (first 5 rows)

Side-by-side view of the **discharge baseline** and the **post-ERW state**.

In [ ]:
for i, row in df_out.head(5).iterrows():
    label     = row['success_label'].upper()
    omega_i   = row['Omega_calcite_init']
    omega_f   = row['Omega_calcite_final']
    si = "supersatd ✓" if omega_i >= 1 else "undersatd"
    sf = "supersatd ✓" if (not np.isnan(omega_f) and omega_f >= OMEGA_THRESH) else "---"
    add_str   = f"{row['rock_add_umol_kg']:.2f} µmol/kg" if row['rock_add_umol_kg'] > 0 else "none needed"

    print(f"╔══ Row {i+1:>4}  [{label}]  {'':═<30}╗")
    print(f"║  Rock: {ROCK_NAME:<12}  Degass: {DEGASS_CASE}  T={row['Temp_C']}°C  S={row['Sal_init']:.4f} PSU")
    print(f"╠─ BASELINE (initial PyCO2SYS discharge run) {'':─<15}╣")
    print(f"║  TA     = {row['TA_init_umol_kg']:>10.2f} µmol/kg    DIC   = {row['DIC_init_umol_kg']:>10.2f} µmol/kg")
    print(f"║  pH     = {row['pH_Total_init']:>10.4f}             pCO₂  = {row['pCO2_init_uatm']:>10.2f} µatm")
    print(f"║  Ca²⁺  = {row['Ca_init_umol_kg']:>10.2f} µmol/kg    CO₃²⁻ = {row['CO3_init_umol_kg']:>10.4f} µmol/kg")
    print(f"║  Ω_calcite  = {omega_i:>8.4f}  → {si}")
    print(f"╠─ POST-ERW {'':─<50}╣")
    print(f"║  Rock added : {add_str}   (j={int(row['j_steps'])}, k={int(row['k_steps'])})")
    print(f"║  TA_final   = {row['TA_final_umol_kg']:>10.2f} µmol/kg    ΔTA   = {row['Delta_TA_umol_kg']:>+10.2f}")
    print(f"║  pH_final   = {row['pH_final']:>10.4f}             ΔpH   = {row['Delta_pH']:>+10.4f}")
    print(f"║  pCO₂_final = {row['pCO2_final_uatm']:>10.2f} µatm        ΔpCO₂ = {row['Delta_pCO2_uatm']:>+10.2f}")
    print(f"║  ΔDIC       = {row['Delta_DIC_umol_kg']:>+10.2f} µmol/kg    ΔCa   = {row['Delta_Ca_umol_kg']:>+10.2f}")
    print(f"║  Ω_calcite_final = {omega_f:>8.4f}  → {sf}")
    print(f"╚{'═'*58}╝")
    print()